# Week 3, Day 3: pandas I: tables of trials

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w003/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import pandas as pd

def make_session(seed=0, n=600, bias=0.05, sigma=0.15, lapse=0.06):
    rng = np.random.default_rng(seed)
    contrasts = np.array([0, 0.0625, 0.125, 0.25, 1.0])
    side = rng.choice([-1, 1], n)                          # -1 left, 1 right
    c = rng.choice(contrasts, n)
    signed = side * c
    p_right = lapse + (1 - 2 * lapse) / (1 + np.exp(-(signed - bias) / sigma))
    choice = np.where(rng.random(n) < p_right, 1, -1)
    timeout = rng.random(n) < 0.04                         # no response on some trials
    rt = 0.25 + 0.6 * np.exp(-3 * c) * rng.lognormal(0, 0.35, n)
    return pd.DataFrame({
        "trial": np.arange(1, n + 1),
        "contrast_left": np.where(side == -1, c, 0.0),
        "contrast_right": np.where(side == 1, c, 0.0),
        "choice": np.where(timeout, 0, choice),            # 0 means no response
        "rt": np.where(timeout, np.nan, rt.round(3)),      # seconds
        "feedback": np.where(timeout, -1, np.where(choice == side, 1, -1)),
    })

df = make_session()
df.head(8)

In [ ]:
print(df.shape)          # (rows, columns)
print(df.dtypes)         # the type of each column
df.describe()            # count, mean, std and quartiles of each numeric column

In [ ]:
df.to_csv("session01_trials.csv", index=False)
again = pd.read_csv("session01_trials.csv")
print(again.equals(df), again.shape)

### Exercise: Easy and hard trials

Make `easy`, the rows where the contrast is 1 on either side. Compute `easy_correct`, the fraction of easy trials with `feedback == 1`, and `zero_correct`, the same for trials with zero contrast on both sides. What should `zero_correct` be if the mouse is guessing?

In [ ]:
easy = ...
easy_correct = ...
zero = ...
zero_correct = ...
print(f"easy: {easy_correct:.2f}   zero contrast: {zero_correct:.2f}")

In [ ]:
def test_easy():
    "easy holds the rows with 100 per cent contrast on either side"
    expect = df[(df["contrast_left"] == 1) | (df["contrast_right"] == 1)]
    assert easy.equals(expect), "Select rows where either contrast column equals 1."

def test_rates():
    "easy_correct and zero_correct are fractions of rewarded trials"
    zero = df[(df["contrast_left"] == 0) & (df["contrast_right"] == 0)]
    assert abs(easy_correct - (easy["feedback"] == 1).mean()) < 1e-12
    assert abs(zero_correct - (zero["feedback"] == 1).mean()) < 1e-12

test_easy()
test_rates()
print("All checks passed")

### Exercise: Signed contrast and clean trials

Add two columns to `df`: `signed_contrast` (right minus left) and `chose_right` (True when `choice == 1`). Then make `clean`: the trials with a reaction time that is present and under 2 seconds.

In [ ]:
df["signed_contrast"] = ...
df["chose_right"] = ...
clean = ...
print(len(df), "trials,", len(clean), "clean")

In [ ]:
import numpy as np
def test_signed():
    "signed_contrast is right minus left"
    assert np.allclose(df["signed_contrast"], df["contrast_right"] - df["contrast_left"])

def test_right():
    "chose_right is True when choice is 1"
    assert df["chose_right"].dtype == bool and (df["chose_right"] == (df["choice"] == 1)).all()

def test_clean():
    "clean has no missing reaction times and no very slow trials"
    assert clean["rt"].notna().all() and (clean["rt"] < 2).all()
    assert len(clean) == ((df["rt"] < 2) & df["rt"].notna()).sum()

test_signed()
test_right()
test_clean()
print("All checks passed")

### Exercise: Bug hunt: the flag that never sets

The code below tries to mark slow trials, but `df['slow']` ends up all False. Selecting rows first and then a column gives a temporary copy, so the assignment is lost. Fix it with a single `.loc[rows, column]` assignment.

In [ ]:
df["slow"] = False
df[df["rt"] > 1]["slow"] = True
print(df["slow"].sum(), "slow trials")

In [ ]:
def test_flag():
    "slow is True exactly for trials with rt over 1 s"
    assert "slow" in df.columns, "Create the column on df itself."
    assert (df["slow"] == (df["rt"] > 1)).all()

test_flag()
print("All checks passed")

In [ ]:
summary = clean.groupby("signed_contrast").agg(
    n=("trial", "size"),
    p_right=("chose_right", "mean"),
    median_rt=("rt", "median"),
)
summary